# Dimensiereductie

Auteurs: Brian van der Bijl (brian.vanderbijl@hu.nl), Tijmen Muller

- Studentnummer: 1833628
- Naam: Mathijs de Jong
- Datum: 06/03/2026

## Deel I: Principal Component Analysis (PCA)

Het _Principal Component Analysis_ (PCA) algoritme kan gebruikt worden om het aantal dimensies van een dataset te reduceren tot de belangrijkste componenten. Als de originele dataset $n$ dimensies heeft, dan kunnen we met onderstaande stappen dit terugbrengen tot een (zelfgekozen) aantal van $n^\prime$ dimensies.

1. Centreer de data.
2. Bereken de covariantie van alle features onderling. 
3. Bereken de Eigenvectors en Eigenvalues van de covariantiematrix.
4. Kies de $n^\prime$ Eigenvectors om de dimensiereductie mee uit te voeren.
5. Vermenigvuldig de $n^\prime$ Eigenvectors met de originele data om de reductie toe te passen.

### Context

Gegeven is een databestand met embeddings van 200 tekstfragmenten. Elke embedding bestaat in 15 dimensies, en is gelabeled met een categorie. We gaan dimensionaliteitsreductie toepassen om de data te kunnen plotten.

De categorie geeft aan in welk genre de tekstfragmenten thuishoren. Daarnaast is onderscheid gemaakt tussen het perspectief waarin het fragment geschreven is: ik (1st person) of hij/haar/hen (3rd person):
- Fantasy (1st person)
- Fantasy (3rd person)
- Science Fiction (1st person)
- Science Fiction (3rd person)
- Romance (1st person)
- Romance (3rd person)
- Crime (1st person)
- Crime (3rd person)

In [351]:
import numpy as np
import pickle
import matplotlib.pyplot as plt

# Show floats on 3 digits, suppress scientific notation
np.set_printoptions(precision=3, suppress=True)

In [352]:
with open('data.pkl', 'rb') as file:
    data = pickle.load(file)

data.sample(5)

e_1,e_2,e_3,e_4,e_5,e_6,e_7,e_8,e_9,e_10,e_11,e_12,e_13,e_14,e_15,label
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
0.324159,0.942513,0.561516,0.966911,1.024427,5.459975,0.997579,1.867844,0.502075,0.919101,1.135698,0.954855,1.359423,1.069583,0.684259,"""Romance (3rd person)"""
0.519645,0.665019,0.576393,1.145396,1.000007,5.249173,0.910483,1.947367,0.565465,1.226234,1.110513,0.996838,1.392525,1.2367,0.339647,"""Romance (3rd person)"""
0.508747,1.364501,0.981314,0.46636,0.385486,3.466497,1.080464,2.127262,1.251522,1.052446,1.198563,0.762797,1.361704,0.761576,1.045253,"""Science Fiction (3rd person)"""
1.508689,0.46763,0.240512,1.010149,0.843448,4.581535,0.289546,2.436911,0.575009,1.303067,0.620383,1.045034,0.426191,0.678861,1.080749,"""Crime (1st person)"""
1.726001,0.887495,0.786724,0.775676,0.966512,3.430738,0.976983,3.369367,1.070422,1.250615,0.687875,1.056892,0.972576,0.71436,0.84539,"""Fantasy (3rd person)"""


### Voorbereidende opdracht

Gegeven een dataset met $m$ datapunten met elk $n$ features en een gewenste reductie tot $n^\prime$ dimensies. Bepaal voor elk van de vijf stappen van het algoritme wat de dimensies (oftewel `shape`) is van de volgende tussenresultaten:

0. De matrix met de originele dataset.
1. De matrix met de gemiddelden per feature om de data mee te centreren.
2. De covariantiematrix.
3. De matrix met de Eigenvectors en de matrix met de Eigenvalues.
4. De matrix met de _geselecteerde_ Eigenvectors.
5. De matrix met de _gereduceerde_ data.

0. M datapunten (rijen) en n features (kolommen) --> de shape is (m, n).
1. Elke feature (kolom) heeft 1 gemiddelde, dus 1 rij. De shape is dus (1, n).
2. De covariantiematrix beschrijft de covariantie tussen alle features onderling. De shape is dus (n, n).
3. De eigenvectors worden berekend met de covariantiematrix. Dit wordt een matrix waarin elke kolom een eigenvector is, met 1 waarde voor elke feature. De shape is dus (n, n).
Voor elke eigenvector is er een eigenvalue, dus n eigenvalues. Dit komt in een vector met shape (n, 1)
4. We hadden de matrix met eigenvectors van shape (n, n). We gooien de kolommen (eigenvectors) weg die niet belangrijk genoeg zijn, en houden n' kolommen over. De shape is dus (n, n').
5. We vermenigvuldigen de originele dataset (m, n) met de geselecteerde eigenvectors (n, n'), dus de shape van het resultaat wordt: (m, n').

### Opdracht 1. Implementatie

Schrijf een eigen implementatie van het PCA-algoritme `compute_pca(X, n_components)` volgens eerdergenoemde stappen van het algoritme. Maak slim gebruik van fucties van `numpy` waar mogelijk, maar zorg wel dat je begrijpt wat je in elke stap doet. De laatste stap is al gegeven in de functiedefinitie hieronder.

Hint: Laat bij stap 3. zien (bijvoorbeeld met een `print()` statement) dat de meest informatieve Eigenvalue al meer dan 50% van de informatie bevat van onze dataset.

#### Input
- `X: numpy.array` - numpy matrix met dimensies $(m, n)$; elke rij is een datapunt in $n$ dimensies
- `n_components: int` - het gewenste aantal dimensies $n^\prime$

#### Output
`X_reduced: numpy.array` - een $(m, n^\prime)$ numpy matrix met de gereduceerde data.

In [353]:
def compute_pca(X, n_components):
    """
    Parameters
    ----------
    X : numpy.ndarray
        Input data matrix of shape (m, n), where m is the number of samples and n is the number of features.
    n_components : int
        The number of principal components (dimensions) to keep.

    Returns
    -------
    X_reduced : numpy.ndarray
        The data projected onto the top n_components principal components.
    """
    arr = np.array(X)

    # 1. Data centreren
    # Centreren: gemiddelde per feature (kolom) berekenen en dat van de kolom af trekken
    gem_per_feature = np.mean(arr, axis=0)
    arr_gecentreerd = arr - gem_per_feature

    # 2. Covariantie-matrix maken
    covariance_matrix = np.cov(arr_gecentreerd, rowvar=False, ddof=1)

    # 3. Eigenvalues en eigenvector van de covariantie-matrix berekenen
    eigenvalues, eigenvectors = np.linalg.eig(covariance_matrix)
    print(f"Eigenvalues van eigenvectors: {eigenvalues.real}")
    print(f"Eigenvectors (kolommen):\n{eigenvectors.real}")

    # 4. De n' beste eigenvectors selecteren om reductie mee te doen
    # De eigenvalues van de eigenvectors beslissen de ranking
    # Eigenvectors sorteren op volgorde van hoogste eigenvalue naar laagste eigenvalue
    indexen_gesorteerd = np.argsort(eigenvalues)    # geeft de indexen die de lijst zouden sorteren
    indexen_gesorteerd = indexen_gesorteerd[::-1]   # lijst omdraaien zodat die van hoog naar laag loopt

    eigenvectors_gesorteerd = eigenvectors[:, indexen_gesorteerd]   # sorteer de kolommen (eigenvectors)

    # De top n' eigenvectors selecteren
    n_eigenvectors = eigenvectors_gesorteerd[:, :n_components]

    # 5. De data reduceren
    return arr_gecentreerd @ n_eigenvectors

#### Test-scenario
Onderstaande code zou de volgende output moeten opleveren (het minteken kan wisselen):

```python
[[ 0.43437323 -0.49820384]
 [ 0.42077249  0.50351448]
 [-0.85514571 -0.00531064]]
 ```

In [354]:
np.random.seed(1)
X = np.random.rand(3, 10)
X_reduced = compute_pca(X, n_components=2)
print(f"Gereduceerde data (output):\n{X_reduced.real}")

Eigenvalues van eigenvectors: [ 0.549  0.251  0.     0.    -0.    -0.    -0.    -0.     0.     0.   ]
Eigenvectors (kolommen):
[[-0.298 -0.002 -0.114  0.633 -0.063 -0.063 -0.03  -0.04  -0.078  0.072]
 [-0.207 -0.038  0.041 -0.652 -0.171 -0.171 -0.053 -0.083  0.102 -0.043]
 [-0.166  0.202  0.277 -0.025 -0.163 -0.163  0.287 -0.003 -0.19   0.162]
 [-0.083  0.574  0.656  0.153  0.119  0.119 -0.304  0.112  0.606 -0.434]
 [-0.615 -0.128 -0.235  0.148  0.549  0.549 -0.429 -0.132 -0.427  0.459]
 [-0.404  0.572 -0.417 -0.254 -0.225 -0.225  0.385 -0.114 -0.182  0.175]
 [ 0.168  0.233 -0.194  0.126  0.215  0.215 -0.44   0.805 -0.304  0.195]
 [ 0.321  0.217 -0.085 -0.128 -0.06  -0.06  -0.45  -0.366 -0.315  0.344]
 [ 0.079 -0.255  0.01   0.017  0.06   0.06   0.179 -0.052 -0.093  0.413]
 [-0.395 -0.345  0.452 -0.176 -0.39  -0.39  -0.247  0.406  0.405 -0.458]]
Gereduceerde data (output):
[[ 0.434 -0.498]
 [ 0.421  0.504]
 [-0.855 -0.005]]


Hierboven zien we dat de eigenvalues van het test-scenario. Het valt op dat de 1e kolom (eigenvector) ongeveer 2/3e van de informatie bevat. De 2e kolom ongeveer 1/3e en de rest niks.

In dit geval is het dus ideaal om naar 2 dimensies te reduceren, zoals in de voorbeeldcode gedaan wordt.

### Opdracht 2. Visualisatie met dimensiereductie

Maak op basis van de aangeleverde `data` een numpy array van de datapunten, en gebruik je PCA-implementatie om een 2D- en 3D-weergave van de data te maken. Maak van elke weergave een plot, waarbij iedere categorie een eigen kleur krijgt.

In [355]:
# TODO: Schrijf hier je code.

In [356]:
# TODO: Schrijf hier je code.

### Opdracht 3. Analyse

Analyseer de resultaten:
1. Welke categorieën zijn op basis van de PCA-reductie te onderscheiden, en welke niet? 
2. Geef aan hoeveel procent van de informatie bewaard is gebleven in 2D en 3D respectievelijk.
3. Hoeveel dimensies zijn nodig om 90% van de informatie te bewaren?
4. En voor 95%?

_Schrijf hier je antwoord._

## Deel II: t-Distributed Stochastic Neighbour Embedding (t-SNE)

Een alternatieve methode voor dimensiereductie is _t-Distributed Stochastic Neighbour Embedding_ (t-SNE). 

### Opdracht 4. Toepassing
Gebruik SciKit-Learn om met behulp van t-SNE de data tot 2 dimensies te reduceren en plot het resultaat (wederom met kleuren voor de categorien). 

In [357]:
# TODO: Schrijf hier je code.

### Opdracht 5. Vergelijking

Vergelijk deze met de resultaten van je PCA-implementatie:

1. Hoe verhoudt de zichtbaarheid van de categorieën zich tussen beide resultaten?
2. Hoe verhouden de algoritmes zich in het behoud van informatie?

Beantwoord de volgende vragen los van de data van deze opdracht:

3. Waarvoor zou je PCA en t-SNE inzetten als je te maken krijgt met een onbekende (mogelijk ongelabelde) dataset?
4. Geef een voorbeeld waar PCA de voorkeur heeft boven t-SNE.
5. Geef een voorbeeld waar t-SNE de voorkeur heeft boven PCA.

_Schrijf hier je antwoord._

### Opdracht 6. Project

Als het goed is, heb je op dit moment een eerste idee van de data waar je in het project mee gaat werken. Geef antwoord op onderstaande vragen.

1. Wat is de dimensionaliteit waar je mee te maken hebt?
2. Beschrijf hoe dimensionaliteitsreductie-algoritmen je kunnen helpen de data te verkennen.

_Schrijf hier je antwoord._